<a href="https://colab.research.google.com/github/krutarth3238/slm-lora-finetuning/blob/main/GPT_2_LoRA_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math

eval_results = trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])
print("Perplexity:", perplexity)

Perplexity: 18.94953466094548


In [7]:
prompt = "My Name is Barry Allen and I moonlight as"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

start = time.time()
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_p=0.9
)
end = time.time()

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print("Generated Text:\n")
print(generated_text)

latency = end - start
tokens_per_sec = 50 / latency

print("\nLatency:", latency)
print("Tokens/sec:", tokens_per_sec)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated Text:

My Name is Barry Allen and I moonlight as a reporter for The New York Times. My wife, Elizabeth, is a journalist at the Times and a longtime contributor to The Washington Post.

Latency: 0.7122418880462646
Tokens/sec: 70.20086973142499


In [5]:
import time



latency = end - start
tokens_per_sec = 100 / latency

print("Latency:", latency)
print("Tokens/sec:", tokens_per_sec)

wandb.log({
    "inference/latency": latency,
    "inference/tokens_per_sec": tokens_per_sec
})


NameError: name 'end' is not defined

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["c_attn"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)


In [4]:
from datasets import load_dataset, concatenate_datasets
from transformers import TrainingArguments, Trainer
import wandb

from peft import LoraConfig, get_peft_model

from datasets import load_dataset, concatenate_datasets

from datasets import load_dataset, concatenate_datasets

from datasets import load_dataset, concatenate_datasets

def load_and_prepare_dataset(
    name,
    subset=None,
    text_column="text",
    split="train",
    sample_size=800
):
    ds = load_dataset(
        name,
        subset,
        split=split,
        keep_in_memory=True
    )

    ds = ds.shuffle(seed=42)
    ds = ds.select(range(min(sample_size, len(ds))))

    if text_column != "text":
        ds = ds.rename_column(text_column, "text")

    ds = ds.remove_columns([c for c in ds.column_names if c != "text"])
    return ds


datasets_list = []

datasets_list.append(load_and_prepare_dataset(
    "wikitext", "wikitext-2-raw-v1", "text"
))

datasets_list.append(load_and_prepare_dataset(
    "wikitext", "wikitext-103-raw-v1", "text"
))

datasets_list.append(load_and_prepare_dataset(
    "roneneldan/TinyStories", None, "text"
))

datasets_list.append(load_and_prepare_dataset(
    "ag_news", None, "text"
))

datasets_list.append(load_and_prepare_dataset(
    "xsum", None, "document"
))

datasets_list.append(load_and_prepare_dataset(
    "cnn_dailymail", "3.0.0", "article"
))

datasets_list.append(load_and_prepare_dataset(
    "squad", None, "context"
))

datasets_list.append(load_and_prepare_dataset(
    "yelp_review_full", None, "text"
))

datasets_list.append(load_and_prepare_dataset(
    "imdb", None, "text"
))

datasets_list.append(load_and_prepare_dataset(
    "Menlo/Instruction-text-only-full", None, "text"
))


combined_dataset = concatenate_datasets(datasets_list)
print("Total combined samples:", len(combined_dataset))

def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = combined_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)


split_dataset = tokenized_dataset.train_test_split(
    test_size=0.1,   # 90% train, 10% validation
    seed=42
)

train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["c_attn"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


training_args = TrainingArguments(
    output_dir="./gpt2-lora-multidata",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    report_to="wandb",
    run_name="gpt2_lora_10datasets"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

trainer.save_model("./final_multidata_model")
tokenizer.save_pretrained("./final_multidata_model")


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 147,456 || all params: 124,587,264 || trainable%: 0.1184


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.923883,2.956282
2,3.195400,2.943539


('./final_multidata_model/tokenizer_config.json',
 './final_multidata_model/tokenizer.json')

# **The whole Process for one Dataset now we move to Multiple Datasets**



In [ ]:
trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")


('./final_model/tokenizer_config.json', './final_model/tokenizer.json')

In [ ]:
import time

start = time.time()
model.generate(**inputs, max_new_tokens=100)
end = time.time()

tokens_generated = 100
latency = end - start
tokens_per_sec = tokens_generated / latency

print("Latency:", latency)
print("Tokens/sec:", tokens_per_sec)

wandb.log({
    "inference/latency": latency,
    "inference/tokens_per_sec": tokens_per_sec
})


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Latency: 1.9998176097869873
Tokens/sec: 50.00456017119061


In [ ]:
prompt = "Hello I am the Flash and"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(output[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Hello I am the Flash and I am the Flash. I am the Flash. I am the Flash. I am the Flash. I am the Flash. I am the Flash. I am the Flash. I am the Flash. I am the Flash. I am the Flash.


In [ ]:
import math

eval_results = trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])
print("Perplexity:", perplexity)


Perplexity: 76.36165635378819


In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments
from transformers import Trainer

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
small_train = dataset["train"].shuffle(seed=42).select(range(5000))
small_val = dataset["validation"].select(range(1000))

def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


tokenized_train = small_train.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_val = small_val.map(tokenize_function, batched=True, remove_columns=["text"])

training_args = TrainingArguments(
    output_dir="./gpt2-lora",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    report_to="wandb",
    run_name="gpt2_lora_wikitext"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val
)

trainer.train()



Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,4.309128,4.349338
2,4.669248,4.335481


TrainOutput(global_step=2500, training_loss=4.672677301025391, metrics={'train_runtime': 565.1319, 'train_samples_per_second': 17.695, 'train_steps_per_second': 4.424, 'total_flos': 1310990008320000.0, 'train_loss': 4.672677301025391, 'epoch': 2.0})

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],  # GPT2 attention layers
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [2]:
import wandb
wandb.login()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: krutarth-a (krutarth-a-kj-somaiya-college-of-engineering) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True